# Context-Aware News Search using ELMo on the Reuters Corpus

<br></br>
<be></br>

## Author: Dr Partha Majumdar
#### ORC-ID: 0009-0002-7375-8034

In [1]:
# !pip install -q transformers==5.0.0 torch==2.10.0+cu128 nltk==3.9.1 pandas==2.2.2 prettytable==3.17.0

In [2]:
import importlib.metadata as im

for pkg in [
    "transformers", "torch", "nltk", "pandas", "prettytable"
]:
    print(pkg, "==", im.version(pkg))

transformers == 5.0.0
torch == 2.10.0+cu128
nltk == 3.9.1
pandas == 2.2.2
prettytable == 3.17.0


In [3]:
import nltk
import torch
import pandas as pd

from nltk.corpus import reuters
from transformers import AutoTokenizer, AutoModel

# STEP 1: Download Reuters

In [4]:
nltk.download("reuters")
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package reuters to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

# STEP 2: Inspect Reuters

In [5]:
print("STEP 2: Reuters Corpus Overview")
print("Number of documents:", len(reuters.fileids()))
print("Sample file IDs:", reuters.fileids()[:10])
print("Sample categories:", reuters.categories()[:20])

STEP 2: Reuters Corpus Overview
Number of documents: 10788
Sample file IDs: ['test/14826', 'test/14828', 'test/14829', 'test/14832', 'test/14833', 'test/14839', 'test/14840', 'test/14841', 'test/14842', 'test/14843']
Sample categories: ['acq', 'alum', 'barley', 'bop', 'carcass', 'castor-oil', 'cocoa', 'coconut', 'coconut-oil', 'coffee', 'copper', 'copra-cake', 'corn', 'cotton', 'cotton-oil', 'cpi', 'cpu', 'crude', 'dfl', 'dlr']


# STEP 3: Select practical categories

In [6]:
TARGET_CATEGORIES = ["earn", "crude", "money-fx", "grain", "interest"]

selected_fileids = []
for fileid in reuters.fileids():
    doc_categories = reuters.categories(fileid)
    if any(cat in TARGET_CATEGORIES for cat in doc_categories):
        selected_fileids.append(fileid)

print("STEP 3: Selected Reuters Subset")
print("Selected documents:", len(selected_fileids))

STEP 3: Selected Reuters Subset
Selected documents: 6113


# STEP 4: Prepare documents

In [7]:
documents = []
metadata = []

for fileid in selected_fileids:
    sentences = reuters.sents(fileid)
    tokens = [token for sent in sentences for token in sent]

    if len(tokens) < 20:
        continue

    text = " ".join(tokens)

    documents.append(text)
    metadata.append({
        "fileid": fileid,
        "categories": reuters.categories(fileid)
    })

print("STEP 4: Prepared Documents")
print("Usable documents:", len(documents))
print("Sample document:", documents[0][:500])
print("Sample metadata:", metadata[0])

STEP 4: Prepared Documents
Usable documents: 6071
Sample document: CHINA DAILY SAYS VERMIN EAT 7 - 12 PCT GRAIN STOCKS A survey of 19 provinces and seven cities showed vermin consume between seven and 12 pct of China ' s grain stocks , the China Daily said . It also said that each year 1 . 575 mln tonnes , or 25 pct , of China ' s fruit output are left to rot , and 2 . 1 mln tonnes , or up to 30 pct , of its vegetables . The paper blamed the waste on inadequate storage and bad preservation methods . It said the government had launched a national programme to re
Sample metadata: {'fileid': 'test/14828', 'categories': ['grain']}


# STEP 5: Load contextual model

In [8]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("STEP 5: Model Loaded")
print("Model name:", MODEL_NAME)
print("Device:", device)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


STEP 5: Model Loaded
Model name: distilbert-base-uncased
Device: cuda


# STEP 6: Embedding helper

In [9]:
def embed_text(text, tokenizer, model, device, max_length=256):
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )

    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)
        token_embeddings = outputs.last_hidden_state
        attention_mask = encoded["attention_mask"].unsqueeze(-1)

        masked_embeddings = token_embeddings * attention_mask
        summed = masked_embeddings.sum(dim=1)
        counts = attention_mask.sum(dim=1)

        sentence_embedding = summed / counts.clamp(min=1e-9)

    return sentence_embedding.squeeze(0).cpu()

# STEP 7: Build document embeddings

In [10]:
document_embeddings = []

print("STEP 7: Building Document Embeddings")
for i, doc in enumerate(documents, start=1):
    embedding = embed_text(doc, tokenizer, model, device)
    document_embeddings.append(embedding)

    if i <= 3:
        print(f"Document {i} embedding shape: {tuple(embedding.shape)}")

print("Total embeddings created:", len(document_embeddings))

STEP 7: Building Document Embeddings
Document 1 embedding shape: (768,)
Document 2 embedding shape: (768,)
Document 3 embedding shape: (768,)
Total embeddings created: 6071


# STEP 8: Similarity

In [11]:
def cosine_similarity(vec1, vec2):
    numerator = torch.dot(vec1, vec2).item()
    denominator = torch.norm(vec1).item() * torch.norm(vec2).item()

    if denominator == 0:
        return 0.0

    return numerator / denominator

# STEP 9: Search function

In [12]:
def semantic_search(query, documents, metadata, document_embeddings, tokenizer, model, device, top_k=5):
    query_embedding = embed_text(query, tokenizer, model, device)

    scored_results = []

    for doc, meta, doc_embedding in zip(documents, metadata, document_embeddings):
        score = cosine_similarity(query_embedding, doc_embedding)
        scored_results.append({
            "score": score,
            "fileid": meta["fileid"],
            "categories": meta["categories"],
            "document": doc
        })

    scored_results = sorted(scored_results, key=lambda x: x["score"], reverse=True)
    return scored_results[:top_k]

# STEP 10: Example queries

In [13]:
queries = [
    "oil production and crude prices",
    "foreign exchange market pressure",
    "grain export demand",
    "interest rate policy by central banks",
    "corporate earnings outlook"
]

print("STEP 10: Search Results")

for query in queries:
    print("\n" + "=" * 100)
    print("QUERY:", query)

    results = semantic_search(
        query=query,
        documents=documents,
        metadata=metadata,
        document_embeddings=document_embeddings,
        tokenizer=tokenizer,
        model=model,
        device=device,
        top_k=3
    )

    for rank, result in enumerate(results, start=1):
        print(f"\nRank {rank}")
        print("Score      :", round(result["score"], 4))
        print("File ID    :", result["fileid"])
        print("Categories :", result["categories"])
        print("Document   :", result["document"][:500], "...")

STEP 10: Search Results

QUERY: oil production and crude prices

Rank 1
Score      : 0.7495
File ID    : test/20944
Categories : ['crude']
Document   : ARCO RAISES CRUDE OIL PRICES 50 CTS BARREL , TODAY , WTI TO 19 . 00 ARCO RAISES CRUDE OIL PRICES 50 CTS BARREL , TODAY , WTI TO 19 . 00 ...

Rank 2
Score      : 0.7468
File ID    : test/16012
Categories : ['corn', 'grain']
Document   : EGYPT SEEKING 500 , 000 TONNES CORN - U . S . TRADERS Egypt is expected to tender April 22 for 500 , 000 tonnes of corn for May through September shipments , private export sources said . ...

Rank 3
Score      : 0.7402
File ID    : training/3169
Categories : ['crude', 'nat-gas']
Document   : HAMILTON OIL & lt ; HAML > SAYS RESERVES RISE Hamilton Oil Corp said reserves at the end of 1986 were 59 . 8 mln barrels of oil and 905 . 5 billion cubic feet of natural gas , or 211 mln barrels equivalent , up 10 mln equivalent barrels from a year before . ...

QUERY: foreign exchange market pressure

Rank 1
Score  

# STEP 11: Tabular output

In [14]:
from prettytable import PrettyTable

table = PrettyTable()

# Define columns
table.field_names = [
    "Query",
    "R",
    "Sc",
    "ID",
    "Cat",
    "Preview"
]

# Set compact column widths (~60 chars)
table.max_width = {
    "Query": 14,
    "R": 2,
    "Sc": 6,
    "ID": 12,
    "Cat": 10,
    "Preview": 16
}

# Alignments
table.align["Query"] = "l"
table.align["Preview"] = "l"
table.align["Cat"] = "l"

# Populate rows
for query in queries:
    results = semantic_search(
        query=query,
        documents=documents,
        metadata=metadata,
        document_embeddings=document_embeddings,
        tokenizer=tokenizer,
        model=model,
        device=device,
        top_k=3
    )

    for rank, result in enumerate(results, start=1):
        table.add_row([
            query if rank == 1 else "",   # ✅ show query only once
            rank,
            round(result["score"], 4),
            result["fileid"],
            ", ".join(result["categories"]),
            result["document"][:120]
        ])

print("STEP 11: Results Table\n")
print(table)

STEP 11: Results Table

+----------------+---+--------+--------------+------------+------------------+
| Query          | R |   Sc   |      ID      | Cat        | Preview          |
+----------------+---+--------+--------------+------------+------------------+
| oil production | 1 | 0.7495 |  test/20944  | crude      | ARCO RAISES      |
| and crude      |   |        |              |            | CRUDE OIL PRICES |
| prices         |   |        |              |            | 50 CTS BARREL ,  |
|                |   |        |              |            | TODAY , WTI TO   |
|                |   |        |              |            | 19 . 00 ARCO     |
|                |   |        |              |            | RAISES CRUDE OIL |
|                |   |        |              |            | PRICES 50 CTS    |
|                |   |        |              |            | BARREL , TODAY , |
|                | 2 | 0.7468 |  test/16012  | corn,      | EGYPT SEEKING    |
|                |   |      